## Pre process functions

In [ ]:
def add_ball_possession(df):
    """
    This function calculates the first ball possession for each team based on specific
    game events. The possession is tracked throughout the game, including overtime periods.

    Parameters:
    -----------
    df : pandas.DataFrame
        The input DataFrame containing game events. It must include the columns 'game_id',
        'period', 'clock', and 'token'.

    Returns:
    --------
    df_modified : pandas.DataFrame
        The original DataFrame with an additional column 'ball_possession' that indicates the
        possession of the ball. Possession is 1 if the home team has the ball, -1 if the away
        team has the ball, and 0 if the possession is neutral or undefined.
    """

    # Tokens associated with the home team having possession
    home_possession_tokens = ['p000_12(1)p454_10(0)', 'p452_10(0)',
                              'p454_10(0)', 'p452_10(1)', 'p454_10(1)']

    # Tokens associated with the away team having possession
    away_possession_tokens = ['p000_12(1)p455_10(0)', 'p453_10(0)',
                              'p455_10(0)', 'p453_10(1)', 'p455_10(1)']

    # Step 1: Identify the initial possession at the start of period 1
    initial_possession = df\
    .query("period == 1 & clock == '12:00:00'")\
     [['game_id', 'period', 'clock', 'token']]\
    .assign(possession = lambda df: np.where(df["token"].isin(home_possession_tokens), 1, 0))\
    .assign(possession = lambda df: np.where(df["token"].isin(away_possession_tokens), -1, df["possession"]))\
    .query("possession != 0")\
    .drop_duplicates(['game_id'])

    # Step 2: Identify possession based on tokens following the initial possession
    following_possession = df\
    .merge(initial_possession[['game_id']].assign(aux = 1), how = 'left')\
    .query("aux != 1")\
    .assign(last_token = lambda x: x['token'].shift(1))\
    .query("last_token == 'p000_12(1)' & clock != '12:00:00'")\
    [['game_id', 'period', 'clock', 'token']]\
    .assign(possession = lambda df: np.where(df["token"].isin(home_possession_tokens), 1, 0))\
    .assign(possession = lambda df: np.where(df["token"].isin(away_possession_tokens), -1, df["possession"]))\
    .query("possession != 0")\
    .drop_duplicates(['game_id'])

    # Combine initial and following possessions
    combined_possession = pd.concat([initial_possession, following_possession])\
    .drop_duplicates(['game_id'])

    # Step 3: Capture possession changes that occur after the initial possession
    additional_possession = df\
    .merge(combined_possession[['game_id']].assign(aux = 1), how = 'left')\
    .query("aux != 1")\
    .assign(last_token = lambda x: x['token'].shift(1))\
    .query("last_token == 'p000_12(1)'")\
    [['game_id', 'period', 'clock', 'token']]\
    .assign(possession = lambda df: np.where(df["token"].isin(['p501_7(4p)']), 1, 0))\
    .assign(possession = lambda df: np.where(df["token"].isin(['p401_7(4p)']), -1, df["possession"]))\
    .query("possession != 0")\
    .drop_duplicates(['game_id'])

    # Combine with previous possessions
    combined_possession = pd.concat([combined_possession, additional_possession])\
    .drop_duplicates(['game_id'])

    # Step 4: Handle possession during overtime periods
    overtime_possession = df\
    .query("period == 5 & clock == '05:00:00'")\
    .assign(possession_ot = lambda df: np.where(df["token"].isin(['p454_10(0)', 'p452_10(0)']), 1, 0))\
    .assign(possession_ot = lambda df: np.where(df["token"].isin(['p455_10(0)', 'p453_10(0)']), -1, df["possession_ot"]))\
    .query("possession_ot != 0")\
    [['game_id', 'period', 'clock', 'token', 'possession_ot']]\
    .drop_duplicates(['game_id'])

    # Combine all possession data, including overtime
    combined_possession = pd.concat([combined_possession, overtime_possession])

    # Step 5: Merge possession data back with the original DataFrame
    df_modified = df\
    .merge(combined_possession, how = 'left')

    # Step 6: Calculate cumulative possession throughout the game
    df_modified['possession'] = df_modified\
    .groupby('game_id', group_keys = True)['possession']\
    .apply(lambda x: x.fillna(0).cumsum())\
    .tolist()

    df_modified['possession_ot'] = df_modified\
    .groupby('game_id', group_keys = True)['possession_ot']\
    .apply(lambda x: x.fillna(0).cumsum())\
    .tolist()

    # Step 7: Final adjustments to possession values
    df_modified = df_modified\
    .assign(possession = lambda df: np.where(df["period"] > 4, 0, df["possession"]))\
    .assign(possession = lambda df: np.where(df["possession"] == 0, df["possession_ot"], df["possession"]))\
    .drop(columns = 'possession_ot')\
    .rename(columns = {'possession': 'ball_possession'})

    return df_modified

def add_indicator_periods(df, column):
    # Create period indicators
    return df\
    .assign(period1 = lambda x: [1 if p == 1 else 0 for p in x[column]])\
    .assign(period2 = lambda x: [1 if p == 2 else 0 for p in x[column]])\
    .assign(period3 = lambda x: [1 if p == 3 else 0 for p in x[column]])\
    .assign(period4 = lambda x: [1 if p == 4 else 0 for p in x[column]])\
    .assign(period_overtime = lambda x: [1 if p > 4 else 0 for p in x[column]])

def categorize_time_bins(period, remaining_time, diff_points):
    """
    Cria:
      - time_bin  : 0..4 (posição dentro do período)
      - period_bin: 0..4 (1Q,2Q,3Q,4Q,OTs)
      - cat_time  : 0..24 = period_bin * 5 + time_bin
    """
    # --- comprimento do período em segundos ---
    period_raw = period.to_numpy()
    # 12 minutos para períodos 1-4, 5 minutos para OT (period > 4)
    period_length = np.where(period_raw > 4, 5 * 600, 12 * 600)

    # --- bins de tempo dentro do período ---
    time_bin = np.zeros(period.shape[0], dtype = int)

    # bin 0: primeiros 60s do período
    mask0 = remaining_time > (period_length - 600)
    time_bin[mask0] = 0

    # bin 1: 2º minuto até faltarem 2min pro fim
    mask1 = (~mask0) & (remaining_time > 1200)
    time_bin[mask1] = 1

    # bin 2: de 2:00 até 1:00
    mask2 = (~mask0) & (~mask1) & (remaining_time > 600)
    time_bin[mask2] = 2

    # bin 3: de 1:00 até 0:20
    mask3 = (~mask0) & (~mask1) & (~mask2) & (remaining_time > 200)
    time_bin[mask3] = 3

    # bin 4: últimos 20 segundos
    mask4 = (~mask0) & (~mask1) & (~mask2) & (~mask3)
    time_bin[mask4] = 4

    # --- bins de período ---
    # 1Q -> 0, 2Q -> 1, 3Q -> 2, 4Q -> 3, OT(s) -> 4
    period_bin = np.where(period_raw > 4, 4, period_raw - 1)

    # --- combinação período × tempo => 0..24 ---
    cat_time = period_bin * 5 + time_bin

 # ---------- DIFERENÇA DE PONTOS ----------
    # ajuste aqui se o nome das colunas for diferente
    score_diff = diff_points.to_numpy()

    diff_bin = np.zeros(period.shape[0], dtype = int)

    # 1) dif <= -20
    diff_bin[score_diff <= -20] = 0

    # 2) -20 < dif < -10
    diff_bin[(score_diff > -20) & (score_diff < -10)] = 1

    # 3) -10 <= dif < -6
    diff_bin[(score_diff >= -10) & (score_diff < -6)] = 2

    # 4) -6 <= dif < -3
    diff_bin[(score_diff >= -6) & (score_diff < -3)] = 3

    # 5) -3 <= dif <= 3
    diff_bin[(score_diff >= -3) & (score_diff <= 3)] = 4

    # 6) 3 < dif <= 6
    diff_bin[(score_diff > 3) & (score_diff <= 6)] = 5

    # 7) 6 < dif <= 10
    diff_bin[(score_diff > 6) & (score_diff <= 10)] = 6

    # 8) 10 < dif < 20
    diff_bin[(score_diff > 10) & (score_diff < 20)] = 7

    # 9) dif >= 20
    diff_bin[score_diff >= 20] = 8
    # ---------- STATE ID FINAL ----------
    # 25 estados de tempo × 9 estados de diferença = 225
    state_id = cat_time * 9 + diff_bin

    return state_id

def build_covariables(dataframe):
    """
    This function transforms the input dataframe by creating and modifying
    several covariates related to fouls, time, periods, and lead status in a game.
    It also includes the addition of ball possession information by using
    the `add_ball_possession` function.

    Parameters:
    dataframe (pd.DataFrame): The input dataframe containing game data.

    Returns:
    pd.DataFrame: The modified dataframe with new and adjusted covariates.
    """

    # Adjust fouls limits for both home (limhfouls) and visiting teams (limvfouls)
    # Set to 0 if fouls >= 1, otherwise 1
    print("Calculating limfouls")

    dataframe = dataframe\
    .assign(ahfbb = lambda df: df['limhfouls'], aafbb = lambda df: df['limvfouls'])\
    .assign(ind_lastplay = lambda df: (df["period"] != df["period"].shift(-1).fillna(0)))\
    .assign(period_ = lambda df: df["period"] + (df["ind_lastplay"] == 1).astype("int"))\
    .assign(limhfouls = lambda df: (df["limhfouls"] < 1).astype("int"))\
    .assign(limvfouls = lambda df: (df["limvfouls"] < 1).astype("int"))\
    .assign(limhtimeout = lambda df: (df["available_hto"] > 0).astype("int"))\
    .assign(limvtimeout = lambda df: (df["available_ato"] > 0).astype("int"))

    clear_output(wait = True)

    # Cap the difference in score (dif) to a range between -30 and 30, then normalize it
    print("Calculating diff points variables")

    dataframe = dataframe\
    .assign(diff = lambda df: df["dif"])\
    .assign(dif = lambda df: np.clip(df["diff"], -30, 30) / 30)\
    .assign(dif_20 = lambda df: np.clip(df["diff"], -20, 20) / 10)\
    .assign(ind_close_3 = lambda df: (np.abs(df["diff"]) <= 3).astype("int"))\
    .assign(ind_close_5 = lambda df: (np.abs(df["diff"]) <= 5).astype("int"))\
    .assign(ind_close_8 = lambda df: (np.abs(df["diff"]) <= 8).astype("int"))\
    .assign(ind_blowout_15 = lambda df: (np.abs(df["diff"]) >= 15).astype("int"))

    clear_output(wait = True)

    # Cap the spent time to a maximum of 300 seconds
    print("Calculating spent time")

    dataframe = dataframe\
    .assign(spent_time = lambda df: np.clip(df["spent_time"], 0, 300))

    clear_output(wait = True)

    # Create a lead indicator: 1 for home team leading the points, -1 for away team leading the points, 0 for tied
    print("Calculating leading variables")

    dataframe = dataframe\
    .assign(lead = lambda df: np.sign(df["dif"]))\
    .assign(home_lead = lambda df: (df["dif"] > 0).astype("int"))\
    .assign(away_lead = lambda df: (df["dif"] < 0).astype("int"))

    clear_output(wait = True)

    # Create indicators for critical time periods in the game
    print("Calculating time and points variables")

    dataframe = dataframe\
    .assign(ind_last_2_minutes = lambda df: ((df["time"] <= 1200) & (df["time"] > 600)).astype("int"))\
    .assign(ind_last_minute = lambda df: ((df["time"] <= 600) & (df["time"] > 200)).astype("int"))\
    .assign(ind_last_20_seconds = lambda df: ((df["time"] <= 200) & (df["time"] > 100)).astype("int"))\
    .assign(ind_last_10_seconds = lambda df: ((df["time"] <= 100) & (df["time"] > 50)).astype("int"))\
    .assign(ind_last_5_seconds = lambda df: ((df["time"] <= 50) & (df["time"] > 10)).astype("int"))\
    .assign(ind_last_second = lambda df: (df["time"] <= 10).astype("int"))\
    .assign(ind_last_2_minutes_game = lambda df: ((df["ind_last_2_minutes"] == 1) & (df["period_"] >= 4)).astype("int"))\
    .assign(ind_last_minute_game = lambda df: ((df["ind_last_minute"] == 1) & (df["period_"] >= 4)).astype("int"))\
    .assign(ind_last_20_seconds_game = lambda df: ((df["ind_last_20_seconds"] == 1) & (df["period_"] >= 4)).astype("int"))\
    .assign(ind_last_10_seconds_game = lambda df: ((df["ind_last_10_seconds"] == 1) & (df["period_"] >= 4)).astype("int"))\
    .assign(ind_last_5_seconds_game = lambda df: ((df["ind_last_5_seconds"] == 1) & (df["period_"] >= 4)).astype("int"))\
    .assign(ind_clutch_2_last_minutes = lambda df: ((df["ind_last_2_minutes_game"] == 1) & (df["ind_blowout_15"] == 0)).astype("int"))\
    .assign(ind_clutch_2_last_minutes_home = lambda df: ((df["ind_clutch_2_last_minutes"] == 1) & (df["home_lead"] == 1)).astype("int"))\
    .assign(ind_clutch_2_last_minutes_away = lambda df: ((df["ind_clutch_2_last_minutes"] == 1) & (df["away_lead"] == 1)).astype("int"))\
    .assign(ind_clutch_last_minute = lambda df: ((df["ind_last_minute_game"] == 1) & (df["ind_close_8"] == 1)).astype("int"))\
    .assign(ind_clutch_last_minute_home = lambda df: ((df["ind_clutch_last_minute"] == 1) & (df["home_lead"] == 1)).astype("int"))\
    .assign(ind_clutch_last_minute_away = lambda df: ((df["ind_clutch_last_minute"] == 1) & (df["away_lead"] == 1)).astype("int"))\
    .assign(ind_clutch_last_20_seconds = lambda df: ((df["ind_last_20_seconds_game"] == 1) & (df["ind_close_5"] == 1)).astype("int"))\
    .assign(ind_clutch_last_20_seconds_home = lambda df: ((df["ind_last_20_seconds_game"] == 1) & (df["home_lead"] == 1)).astype("int"))\
    .assign(ind_clutch_last_20_seconds_away = lambda df: ((df["ind_last_20_seconds_game"] == 1) & (df["away_lead"] == 1)).astype("int"))\
    .assign(ind_clutch_last_10_seconds = lambda df: ((df["ind_last_10_seconds_game"] == 1) & (df["ind_close_3"] == 1)).astype("int"))\
    .assign(ind_clutch_last_10_seconds_home = lambda df: ((df["ind_last_10_seconds_game"] == 1) & (df["home_lead"] == 1)).astype("int"))\
    .assign(ind_clutch_last_10_seconds_away = lambda df: ((df["ind_last_10_seconds_game"] == 1) & (df["away_lead"] == 1)).astype("int"))\
    .assign(ind_clutch_last_5_seconds = lambda df: ((df["ind_last_5_seconds_game"] == 1) & (df["ind_close_3"] == 1)).astype("int"))\
    .assign(ind_clutch_last_5_seconds_home = lambda df: ((df["ind_last_5_seconds_game"] == 1) & (df["home_lead"] == 1)).astype("int"))\
    .assign(ind_clutch_last_5_seconds_away = lambda df: ((df["ind_last_5_seconds_game"] == 1) & (df["away_lead"] == 1)).astype("int"))\
    .assign(ind_blowout_last_minute = lambda df: ((df["ind_last_minute_game"] == 1) & (df["ind_blowout_15"] == 1)).astype("int"))

    clear_output(wait = True)

    # Normalize the remaining time in the game
    print("Calculating time variables")

    dataframe = dataframe\
    .assign(last_time = lambda df: 1 - df['last_time'] / 7200,
            time = lambda df: 1 - df['time'] / 7200)

    clear_output(wait = True)

    # Create period indicators
    print("Calculating period variables")

    dataframe = dataframe\
    .pipe(func = add_indicator_periods, column = 'period_')

    clear_output(wait = True)

    # Add ball possession data to the dataframe
    print("Calculating add ball possession variables")

    dataframe = dataframe\
    .pipe(func = add_ball_possession)

    clear_output(wait = True)

    print("Calculating final variables")
    #     .assign(limhtimeout = lambda df: np.where(df["limhtimeout"] == 1, 0, df["limhtimeout"]))\
    # .assign(limvtimeout = lambda df: np.where(df["limvtimeout"] == 1, 0, df["limvtimeout"]))\

    dataframe = dataframe\
    .assign(time = lambda df: np.where(df["ind_lastplay"] == 1, 0, df["time"]))\
    .assign(limhfouls = lambda df: np.where(df["ind_lastplay"] == 1, 0, df["limhfouls"]))\
    .assign(limvfouls = lambda df: np.where(df["ind_lastplay"] == 1, 0, df["limvfouls"]))\
    .assign(ind_last_2_minutes = lambda df: np.where(df["ind_lastplay"] == 1, 0, df["ind_last_2_minutes"]))\
    .assign(ind_last_minute = lambda df: np.where(df["ind_lastplay"] == 1, 0, df["ind_last_minute"]))\
    .assign(ind_last_20_seconds = lambda df: np.where(df["ind_lastplay"] == 1, 0, df["ind_last_20_seconds"]))\
    .assign(ind_last_10_seconds = lambda df: np.where(df["ind_lastplay"] == 1, 0, df["ind_last_10_seconds"]))\
    .assign(ind_last_5_seconds = lambda df: np.where(df["ind_lastplay"] == 1, 0, df["ind_last_5_seconds"]))\
    .assign(ind_last_second = lambda df: np.where(df["ind_lastplay"] == 1, 0, df["ind_last_second"]))\
    .assign(ind_last_2_minutes_game = lambda df: np.where(df["ind_lastplay"] == 1, 0, df["ind_last_2_minutes_game"]))\
    .assign(ind_last_minute_game = lambda df: np.where(df["ind_lastplay"] == 1, 0, df["ind_last_minute_game"]))\
    .assign(ind_last_20_seconds_game = lambda df: np.where(df["ind_lastplay"] == 1, 0, df["ind_last_20_seconds_game"]))\
    .assign(ind_last_10_seconds_game = lambda df: np.where(df["ind_lastplay"] == 1, 0, df["ind_last_10_seconds_game"]))\
    .assign(ind_last_5_seconds_game = lambda df: np.where(df["ind_lastplay"] == 1, 0, df["ind_last_5_seconds_game"]))\
    .assign(remaining_time = lambda df: df["clock"].str.split(":").apply(lambda x: np.asarray(x, dtype = "float") * [600, 10, 0.1]).apply(lambda x: sum(x)).astype("int"))

    clear_output(wait = True)
    print("It's over")

    return dataframe

def assign_data_split(dataframe, seed = 42):
    # split data into train, dev and test functions
    """
    Assigns each row in the dataframe to a train, dev, or test split based on
    random assignment.

    Args:
        dataframe (pd.DataFrame): The dataframe to split, containing at least
        the 'game_id' column.

    Returns:
        pd.DataFrame: A dataframe with an additional 'split' column indicating
        the data split.
    """
    # Ensure randomness is reproducible
    np.random.seed(seed)

    # Create a 'split' column directly within the DataFrame
    dataframe['split'] = np.random.rand(len(dataframe))
    dataframe['split'] = dataframe['split']\
    .apply(lambda x: 'train' if x <= 0.8 else 'dev' if x <= 0.9 else 'test')

    return dataframe

def normalize_boxscore_variables(df):
    df_ = df.copy()
    # FGM > [36, 45]
    df_[['home_FGM', 'away_FGM']] = (df_[['home_FGM', 'away_FGM']] - 36)/(45 - 36)
    # FGA > [82, 95]
    df_[['home_FGA', 'away_FGA']] = (df_[['home_FGA', 'away_FGA']] - 82)/(95 - 82)
    # FGP > [0.42, 0.51]
    df_[['home_FGP', 'away_FGP']] = (df_[['home_FGP', 'away_FGP']] - 0.42)/(0.51 - 0.42)
    # 3FGM > [8, 17]
    df_[['home_3FGM', 'away_3FGM']] = (df_[['home_3FGM', 'away_3FGM']] - 8)/(17 - 8)
    # 3FGA > [25, 44]
    df_[['home_3FGA', 'away_3FGA']] = (df_[['home_3FGA', 'away_3FGA']] - 25)/(44 - 25)
    # 3FGP > [0.3, 0.4]
    df_[['home_3FGP', 'away_3FGP']] = (df_[['home_3FGP', 'away_3FGP']] - 0.3)/(0.4 - 0.3)
    # FTM > [13, 23]
    df_[['home_FTM', 'away_FTM']] = (df_[['home_FTM', 'away_FTM']] - 13)/(23 - 13)
    # FTA > [18, 28]
    df_[['home_FTA', 'away_FTA']] = (df_[['home_FTA', 'away_FTA']] - 18)/(28 - 18)
    # FTP > [0.7, 0.85]
    df_[['home_FTP', 'away_FTP']] = (df_[['home_FTP', 'away_FTP']] - 0.7)/(0.85 - 0.7)
    # RebOff > [8, 14]
    df_[['home_RebOff', 'away_RebOff']] = (df_[['home_RebOff', 'away_RebOff']] - 8)/(14 - 8)
    # RebDef > [30, 40]
    df_[['home_RebDef', 'away_RebDef']] = (df_[['home_RebDef', 'away_RebDef']] - 30)/(40 - 30)
    # RebTot > [40 , 50]
    df_[['home_RebTot', 'away_RebTot']] = (df_[['home_RebTot', 'away_RebTot']] - 40)/(50 - 40)
    # AST > [20, 30]
    df_[["home_AST", "away_AST"]] = (df_[["home_AST", "away_AST"]] - 20)/(30 - 20)
    # STL > [5, 10]
    df_[["home_STL", "away_STL"]] = (df_[["home_STL", "away_STL"]] - 5)/(10 - 5)
    # BLK > [3, 7]
    df_[["home_BLK", "away_BLK"]] = (df_[["home_BLK", "away_BLK"]] - 3)/(7 - 3)
    # TO > [10, 17]
    df_[["home_TO", "away_TO"]] = (df_[["home_TO", "away_TO"]] - 10)/(17 - 10)
    # FP > [17, 23]
    df_[["home_FP", "away_FP"]] = (df_[["home_FP", "away_FP"]] - 17)/(23 - 17)
    # Points > [100, 124]
    df_[["home_points", "away_points"]] = (df_[["home_points", "away_points"]] - 100)/(124 - 100)
    # PMP > [-10, 10]
    df_[["home_PMP", "away_PMP"]] = (df_[["home_PMP", "away_PMP"]] - (-10))/(10 - (-10))

    #    df_[["home_outcome", "away_outcome"]] = (df_[["home_outcome", "away_outcome"]] - (-10))/(10 - (-10))
    return df_

def put_categorical_training_outcomes(df):

    return df\
    .assign(ctgc = 0)\
    .assign(ctgc = lambda df: np.where((df["ctgc"] == 0) & (df["home_outcome"] >= 0.5) & (df["away_outcome"] >= 0.5), 2, df["ctgc"]))\
    .assign(ctgc = lambda df: np.where((df["ctgc"] == 0) & (df["home_outcome"] >= 0.5) & (df["away_outcome"] <  0.5), 3, df["ctgc"]))\
    .assign(ctgc = lambda df: np.where((df["ctgc"] == 0) & (df["home_outcome"] <  0.5) & (df["away_outcome"] >= 0.5), 4, df["ctgc"]))\
    .assign(ctgc = lambda df: np.where((df["ctgc"] == 0) & (df["home_outcome"] <  0.5) & (df["away_outcome"] <  0.5), 5, df["ctgc"]))\
    .assign(ctgc = lambda df: np.where((df["home_outcome"] >=  0.4) & (df["home_outcome"] <=  0.6) & (df["away_outcome"] >=  0.4) & (df["away_outcome"] <=  0.6), 1, df["ctgc"]))

def build_token_tensors(df, token_context = 10, add_box_score_features = True):

    df["token_id"] = df["token"].map(token_to_integer).astype(int)

    g = df.groupby("game_id")["token_id"]

    # 2) criar as 10 colunas shiftadas de uma vez (t-10 ... t-1)
    shifts = [g.shift(i).fillna(0).astype(int) for i in range(token_context, 0, -1)]

    # 3) empilhar em um array (n_linhas, 10) e transformar em lista de listas
    df["token_features"] = np.stack(shifts, axis = 1).tolist()

    game_variables = ['period1', 'period2', 'period3', 'period4', 'period_overtime', 'time',
                      'limhfouls', 'limvfouls', 'limhtimeout', 'limvtimeout', 'dif', 'dif_20',
                      'ind_close_3', 'ind_close_5', 'ind_close_8', 'ind_blowout_15',
                      'ind_last_2_minutes', 'ind_last_minute', 'ind_last_20_seconds',
                      'ind_last_10_seconds', 'ind_last_5_seconds', 'ind_last_second',
                      'lead', 'home_lead', 'away_lead', 'ind_last_2_minutes_game',
                      'ind_last_minute_game', 'ind_last_20_seconds_game',
                      'ind_last_10_seconds_game', 'ind_last_5_seconds_game',
                      'ind_clutch_2_last_minutes', 'ind_clutch_2_last_minutes_home',
                      'ind_clutch_2_last_minutes_away', 'ind_clutch_last_minute',
                      'ind_clutch_last_minute_home', 'ind_clutch_last_minute_away',
                      'ind_clutch_last_20_seconds', 'ind_clutch_last_20_seconds_home',
                      'ind_clutch_last_20_seconds_away', 'ind_clutch_last_10_seconds',
                      'ind_clutch_last_10_seconds_home', 'ind_clutch_last_10_seconds_away',
                      'ind_blowout_last_minute', 'ball_possession']

    list_variables1 = ["period1", "ind_close_3", "ind_close_5", "ind_close_8"]
    list_variables2 = ["period2", "period3", "period4", "period_overtime", "time",
                       "limhfouls", "limvfouls", "limhtimeout", "limvtimeout", "dif",
                       "dif_20", "ind_blowout_15", "ind_last_2_minutes", "ind_last_minute",
                       "ind_last_20_seconds", "ind_last_10_seconds", "ind_last_5_seconds",
                       "ind_last_second", "lead", "home_lead", "away_lead", "ind_last_2_minutes_game",
                       "ind_last_minute_game", "ind_last_20_seconds_game", "ind_last_10_seconds_game",
                       "ind_last_5_seconds_game", "ind_clutch_2_last_minutes",
                       "ind_clutch_2_last_minutes_home", "ind_clutch_2_last_minutes_away",
                       "ind_clutch_last_minute", "ind_clutch_last_minute_home",
                       "ind_clutch_last_minute_away", "ind_clutch_last_20_seconds",
                       "ind_clutch_last_20_seconds_home", "ind_clutch_last_20_seconds_away",
                       "ind_clutch_last_10_seconds", "ind_clutch_last_10_seconds_home",
                       "ind_clutch_last_10_seconds_away", "ind_blowout_last_minute",
                       "ball_possession"]

    df[game_variables] = (
        df.groupby("game_id")[game_variables]
        .shift(1)
    )

    df[list_variables1] = df[list_variables1].fillna(1)
    df[list_variables2] = df[list_variables2].fillna(0)

    df["remaining_time"] = df.groupby("game_id")["remaining_time"].shift(1).fillna(7200)
    df["diff"] = df.groupby("game_id")["diff"].shift(1).fillna(0)

    df["game_numerical_features"] = df[game_variables].to_numpy().tolist()

    if add_box_score_features:
        complete_numerical_vector = np.asarray(df["game_numerical_features"].to_list()), np.asarray(df["boxscore_list"].to_list())
        complete_numerical_vector = np.hstack(complete_numerical_vector)
    else:
        complete_numerical_vector = np.asarray(df["game_numerical_features"].to_list())

    df["numerical_features"] = list(complete_numerical_vector)

    df["target_tokens"] = df["token_id"].to_list()

    return df

def build_time_tensors(df, token_context = 10, add_box_score_features = True):

    df["token_id"] = df["token"].map(token_to_integer).astype(int)

    g = df.groupby("game_id")["token_id"]

    # 2) criar as 10 colunas shiftadas de uma vez (t-10 ... t-1)
    shifts = [g.shift(i).fillna(0).astype(int) for i in range(token_context - 1, -1, -1)]

    # 3) empilhar em um array (n_linhas, 10) e transformar em lista de listas
    df["token_features"] = np.stack(shifts, axis = 1).tolist()

    game_variables = ['period1', 'period2', 'period3', 'period4', 'period_overtime', 'last_time',
                      'limhfouls', 'limvfouls', 'limhtimeout', 'limvtimeout', 'dif', 'dif_20',
                      'ind_close_3', 'ind_close_5', 'ind_close_8', 'ind_blowout_15',
                      'ind_last_2_minutes', 'ind_last_minute', 'ind_last_20_seconds',
                      'ind_last_10_seconds', 'ind_last_5_seconds', 'ind_last_second',
                      'lead', 'home_lead', 'away_lead', 'ind_last_2_minutes_game',
                      'ind_last_minute_game', 'ind_last_20_seconds_game',
                      'ind_last_10_seconds_game', 'ind_last_5_seconds_game',
                      'ind_clutch_2_last_minutes', 'ind_clutch_2_last_minutes_home',
                      'ind_clutch_2_last_minutes_away', 'ind_clutch_last_minute',
                      'ind_clutch_last_minute_home', 'ind_clutch_last_minute_away',
                      'ind_clutch_last_20_seconds', 'ind_clutch_last_20_seconds_home',
                      'ind_clutch_last_20_seconds_away', 'ind_clutch_last_10_seconds',
                      'ind_clutch_last_10_seconds_home', 'ind_clutch_last_10_seconds_away',
                      'ind_blowout_last_minute', 'ball_possession']

    list_variables1 = ["period1", "ind_close_3", "ind_close_5", "ind_close_8"]
    list_variables2 = ["period2", "period3", "period4", "period_overtime", "last_time",
                       "limhfouls", "limvfouls", "limhtimeout", "limvtimeout", "dif",
                       "dif_20", "ind_blowout_15", "ind_last_2_minutes", "ind_last_minute",
                       "ind_last_20_seconds", "ind_last_10_seconds", "ind_last_5_seconds",
                       "ind_last_second", "lead", "home_lead", "away_lead", "ind_last_2_minutes_game",
                       "ind_last_minute_game", "ind_last_20_seconds_game", "ind_last_10_seconds_game",
                       "ind_last_5_seconds_game", "ind_clutch_2_last_minutes",
                       "ind_clutch_2_last_minutes_home", "ind_clutch_2_last_minutes_away",
                       "ind_clutch_last_minute", "ind_clutch_last_minute_home",
                       "ind_clutch_last_minute_away", "ind_clutch_last_20_seconds",
                       "ind_clutch_last_20_seconds_home", "ind_clutch_last_20_seconds_away",
                       "ind_clutch_last_10_seconds", "ind_clutch_last_10_seconds_home",
                       "ind_clutch_last_10_seconds_away", "ind_blowout_last_minute",
                       "ball_possession"]

    df[list_variables1] = df[list_variables1].fillna(1)
    df[list_variables2] = df[list_variables2].fillna(0)

    df["remaining_time"] = df.groupby("game_id")["remaining_time"].shift(1).fillna(7200)

    df["game_numerical_features"] = df[game_variables].to_numpy().tolist()

    if add_box_score_features:
        complete_numerical_vector = np.asarray(df["game_numerical_features"].to_list()), np.asarray(df["boxscore_list"].to_list())
        complete_numerical_vector = np.hstack(complete_numerical_vector)
    else:
        complete_numerical_vector = np.asarray(df["game_numerical_features"].to_list())
    df["numerical_features"] = list(complete_numerical_vector)

    df["target_time"] = df["spent_time"].to_list()

    return df
